## Continuous Integration with GitHub Actions

In this notebook, we look at how Continuous Integration (CI) has been set up on GitHub for `Flixtube`.

**Note that you must have completed the previous notebook (01_github_monorepo) before working through this notebook.**

---

## Examine the GitHub Workflows

- Visit `https://github.com/<YourGitHubAccount>/monorepo/actions`
  - Wait until all workflows turn green (about 8 minutes from when the repository was created).
  - Notice the workflows in the left margin.
    - Click `Show more workflows...` to show all workflows.
    - Notice the `Integrate <Microservice>` workflows in the left margin (this is the automated CI part of the CI/CD  pipepline).
    - Notice the `Deploy <Microservice>` workflows in the left margin (this is the manual CD part of the CI/CD pipepline).

### The Continuous Integration (CI) Workflow for `Flixtube.Web`

- Let's examine the `Integrate Flixtube.Web` workflow.
  - Click on `Integrate Flixtube.Web` in the left margin.
  - Then click the `web-ci.yaml` link in the top left, which will take you to the `Code`.
  - Notice in the left margin that this file is located in the subfolder `.github/workflows`.
    - **All GitHub workflows have to be located in this subfolder**.
  - In the main area, the contents of the file is displayed.

- Let's open the file in VSCode instead.
  - Right-click the file `.github/workflows/web-ci.yaml`, choose `Open to the side` and look at the various commads in the file.
  - `name` gives the workflow a name (displayed under GitHub Actions).
  - `on` controls if and when the workflow is triggered.
    - `push` means the workflow is triggered on a push.
      - `branches` contains a list of branches pushed to in order to trigger the workflow.
      - `paths` contains a list of repository paths with changed files in order to trigger the workflow.
      - So, with the current settings, when a `push` is done to the `main` branch and a file has been modified under the `Flixtube.Web` folder (or subfolders), this workflow will be triggerd.
    - `pull_request` means the workflow is triggered on a pull request.
      - The other settings are the same as for the `push`.
    - `workflow_dispatch` means this workflow can be triggered manually from GitHub's GUI.
      - Click the `Integrate Flixtube.Web` workflow under `Actions` on GitHub.
      - Notice the `Run workflow` combobox in the right of the main area.
  - `jobs` is used to define one or more jobs (a workflow is made up of one or more jobs).
    - `integrate` is the name of the one and only job defined in this workflow.
      - `runs-on` determines what image is used to execute the *runner* on.
        - The current setting `ubuntu-latest` means an Ubuntu-based container will be used to run all commands in.
      - `env` is used to define environment variables available to all `steps` in the job.
        - In this case the two environment variables `NAME` and `CONTAINER_NAME` are defined.
      - `steps` contains a list of steps (tasks) that will be executed sequentailly.
        - `name` is used to name a step.
        - `uses` executes a pre-defined action.
          - Most pre-defined actions can be found here: https://github.com/actions
          - `with` configures a pre-defined action with name-value pairs specific to the action.
        - `run` is used to run a command in the container (based on `ubuntu-latest` in this case).
      - The current steps do the following.
        - `uses: actions/checkout@v4` (https://github.com/actions/checkout) checks out the GitHub repository to the *runner's* container (i.e. copies the GitHub repository to the container based on `ubuntu-latest`).
        - `uses: actions/setup-dotnet@v4` (https://github.com/actions/setup-dotnet) is commented out, but if you needed to install the dotnet runtime in the *runner's* container, your would use this action.
          - `with` can be used to configure the dotnet version.
            - In this case `dotnet-version` is set to `9.0.101`.
        - `uses: docker/setup-buildx-action@v2` (https://github.com/docker/setup-buildx-action) installs Docker in the *runner's* container.
        - `run: chmod +x ./scripts/cicd/test.sh && ./scripts/cicd/test.sh` executes the given command in the *runner's* container.
          - This will make the script `test.sh` executable.
          - Then the script `test.sh` is executed in the *runner's* container.
          - If you look at the file `scripts/cicd/test.sh` you'll see that:
            - It uses the environment variables `NAME` and `CONTAINER_NAME` (set above under `env`).
            -  It runs Docker Compose to start the microservice `Flixtube.Web` and all of its dependencies.
            -  Then it invokes `dotnet test` to run all tests.
            -  It stores screenshots from the Playwright end-to-end tests in the folder `./screenshots` in the *runner's* container.
            - Finally, it uses Docker Compose to stop the microservice and all of its dependencies.
        - `uses: actions/upload-artifact@v4` (https://github.com/actions/upload-artifact) uploads an artifact from the *runner's* conatiner to GitHub Actions.
          - In this case, it uploads the folder `./screenshots` in the *runner's* container under the artifact name `screenshots`.
          - This is what you see under the workflow on GitHub.

- **Note that the other CI workflow files have a similar structure.**

---

## Triggering a CI Workflow

Let's examine how we can trigger one of the CI workflows, in this case the `Integrate Flixtube.Metadata` workflow.

- In your GitHub repository, click the `Actions` tab.
- In the left margin, click `Show more workflows...` to show all workflows.
- In the left margin, click the `Integrate Flixtube.Metadata` workflow.
- Click `metadata-ci.yaml` at the top of the main area to quickly peruse the workflow.
  - Notice it has exactly the same structure as the `Flixtube.Web` CI workflow examined above.
  - The only difference is:
    - What subfolder triggers the workflow, i.e. `Flixtube.Metadata/**` under `push` and `pull_request`.
    - The environment variables' values (`NAME: metadata` and `CONTAINER_NAME: metadata`).
    - No artifact is created (for `Flixtube.Web` above screenshots from the end-to-end tests were published as a GitHub artifact).
  - Notice that the workflow can be triggered in three different ways:
    - `on.push.branches` is set to `main` with `on.push.paths` set to `Flixtube.Metadata/**`, which will trigger the workflow:
      - When a `push` is made to the `main` branch and any file below the `Flixtube.Metadata/**` folder has been modified.
    - `on.pull_request.branches` is set to `main` with `on.pull_request.paths` set to `Flixtube.Metadata/**`, which triggers:
      - When a `pull_request` is made to the `main` branch and any file below the `Flixtube.Metadata/**` folder has been modified.
    - `workflow_dispatch` which adds the possibility of manually triggering the workflow via GitHub's user interface.
      - Hit the back button in your browser, and notice the `Run workflow` drop-down listbox in the far right.
      - Click the down arrow.
      - Notice the `Run workflow` button which will trigger the workflow (on the branch selected in the drop-down listbox).



### Triggering the `Integrate Flixtube.Metadata` Workflow via a `push`

- In VSCode's Explorer, expand `monorepo -> Flixtube.Metadata -> Flixtube.Metadata`.
- Right-click `Program.cs` and choose `Open to the side`.
- Add any comment to the top of the file e.g. `//Testing CI push`.
- Save the file.
- Switch to the Source Control View (Windows/Linux: `Ctrl + Shift + G + G`, Mac: `Cmd + Shift + G + G`).
- Expand `monorepo`.
- Stage the `Program.cs` file by clicking the `+` icon next to it.
- Enter a commit message into the textbox, e.g. `add comment`
- Click the down-arrow next to the `Commit` button and choose `Commit & push`.
- On GitHub, for the `monorepo` repository, click the `Actions` tab.
- Notice the `Integrate Flixtube.Metadata` workflow has been triggered with the heading `add comment` (or whatevery comment you provided).
- Click the workflow instance `add comment` (or whatevery comment you provided).
- Click the `Integrate` job.
- Notice the `steps` for the `Integrate` job are displayed.
- You can expand any `step` to see the output from that step.
  - For example, expand the `Test` step and scroll to the bottom (you might have to wait a while for the step to complete).
    - Notice all the tests have been run (and passed).
      - `Flixtube.Metadata.UnitTests.dll` has 4 out of 4 passing tests.
      - `Flixtube.Metadata.IntegrationTests.dll` has 4 out of 4 passing tests.
- In the left margin, notice a green icon next to the job name `integrate`.
  - This means the entire workflow executed without any errors, that all the tests passed, and that the code has been merged into `main`.
  - A red icon would mean at least one of the steps failed, and no code has been merged into `main`.
- Click the `Code` tab at the top of the web page.
- Expand `Flixtube.Metadata -> Flixtube.Metadata`.
- Click on `Program.cs` and notice your comment at the top of the file (which means the code change was merged into `main`).

### Triggering the `Integrate Flixtube.Metadata` Workflow Manually

- On GitHub, for the `monorepo` repository, click the `Actions` tab.
- In the left margin, click `Show more workflows...` (to show all workflows).
- Click `Integrate Flixtube.Metadata`.
- In the far right, click the `Run workflow` drop-down listbox.
- Click the `Run workflow` button.
- Click the `Actions` tab, and notice the workflow has been triggered with the heading `Integrate Flixtube.Metadata`.
  - The orange icon will keep rotating as the workflow is run.
  - It will turn green if the workflow runs without any errors and all tests have passed.
  - It will turn red if any step fails.